# black-box regression results: Pareto plots

Generates the accuracy/simplicity Pareto figures for the black-box regression problems.

In [ ]:
import pandas as pd
import json
import os
import numpy as np; np.random.seed(42)
from glob import glob
from tqdm import tqdm
import seaborn as sns
import pdb

import sys; sys.path.append('..')

import matplotlib.pyplot as plt
import matplotlib

matplotlib.rc('pdf', fonttype=42)
matplotlib.rc('ps', fonttype=42)

params = {'mathtext.default': 'regular' }          
plt.rcParams.update(params)

sns.set(font_scale=1.0, style='whitegrid')
sns.set(palette='magma')
sns.set_style("whitegrid")

plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Nimbus Roman', 'Times New Roman', 'Liberation Serif', 'DejaVu Serif'],
    'font.size': 11,
    'axes.titlesize': 11,
    'axes.labelsize': 11,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11,
    'legend.title_fontsize': 11,
})

def is_rasr(name):
    return 'RASR' in name

NSR_METHODS = {'E2E', 'SNIP', 'NeSymRes', 'TPSR', 'Flash-ANSR'}  # uDSR excluded: retrained per dataset, so not amortized
def is_nsr(name):
    return name in NSR_METHODS or is_rasr(name)
NSR_MARKER = '■'
NSR_MARKER_COLOR = '#2a78d6'

if True: # Off-the-shelf evaluation
    rdir  = '../../results/black-box/'
    figdir = f'../figs/black-box/'
else: # with HP tuning
    rdir  = '../../results/black-box-tuned/'
    figdir = f'../figs/black-box-tuned/'

datadir = '../../datasets/blackbox/'

pretty_labels = {'model_size': 'Mean Expression Size', 'r2_test_rank': 'Mean Test $R^2$ Rank', 'r2_test_norm': 'Mean Normalized $R^2$'}

def save(name='tmp',h=None):
    name = name.strip().replace(' ','-').replace('%','pct')
    if h == None:
        h = plt.gcf()

    h.tight_layout()

    print('saving',figdir+'/'+name+'.pdf')

    if not os.path.exists(figdir):
        os.makedirs(figdir)

    plt.savefig(figdir+'/'+name+'.pdf', bbox_inches='tight')


# read data from feather

In [ ]:
df_results = pd.read_feather(rdir+'results.feather')
df_sum = pd.read_csv(rdir+'results-summary.csv.gz', compression='gzip')

# -1 marks a failed complexity extraction; mask it to NaN so means/medians skip it
df_results['model_size'] = df_results['model_size'].replace(-1, np.nan)
df_sum['model_size'] = df_sum['model_size'].replace(-1, np.nan)

df_results = df_results[~df_results['algorithm'].isin(['Genetic Engine rs', 'Genetic Engine hc', 'Genetic Engine 1p1'])]
df_sum = df_sum[~df_sum['algorithm'].isin(['Genetic Engine rs', 'Genetic Engine hc', 'Genetic Engine 1p1'])]

df_results = df_results.replace({'Genetic Engine': 'G. Engine'})
df_sum = df_sum.replace({'Genetic Engine': 'G. Engine'})

print('mean trial count:')
print(df_results.groupby('algorithm')['dataset'].count().sort_values()
      / df_results.dataset.nunique())

In [ ]:
from scripts.impute_missing_runs import impute_missing_runs

_impute_missing_runs = True
_impute_r2_source     = 'fixed'  # 'fixed' or 'linear'
_impute_r2_test       = 0.0
_impute_model_size    = "dataset_median"  # 'dataset_median', 'algorithm_median', 'algorithm_mean', 'linear_expression_size', or a fixed number
_impute_algorithms    = None
_linear_algorithm_name = 'Linear'

if _impute_missing_runs:
    df_results, df_sum = impute_missing_runs(
        df_results, df_sum,
        datadir=datadir,
        impute_r2_source=_impute_r2_source,
        impute_r2_test=_impute_r2_test,
        impute_model_size=_impute_model_size,
        impute_algorithms=_impute_algorithms,
        linear_algorithm_name=_linear_algorithm_name,
    )


In [ ]:
_r2_norm_strategy = 'median'  # 'minmax': (method - worst) / (best - worst)
                               # 'median': (method - median) / (best - median)
_r2_norm_clip_negative = True  # clip normalized scores below 0 to 0 (keeps the scale in [0, 1] with the median strategy)
_R2_NORM_MIN_DENOM = 1e-5
_IMPROVABILITY_MIN_ERR = 1e-12

def _normalize_r2(x, strategy, clip_negative=None):
    clip_negative = _r2_norm_clip_negative if clip_negative is None else clip_negative
    best = x.max()
    baseline = x.min() if strategy == 'minmax' else x.median()
    denom = best - baseline
    if denom == 0:
        return pd.Series(1.0, index=x.index)
    norm = (x - baseline) / max(denom, _R2_NORM_MIN_DENOM)
    return norm.clip(lower=0) if clip_negative else norm

def _improvability(err):
    """Relative headroom to the best method in percent, (err - best_err) / err * 100 (lower is better)."""
    best = err.min()
    return ((err - best) / err.clip(lower=_IMPROVABILITY_MIN_ERR) * 100).clip(lower=0, upper=100)

def _apply_excluding_similar(group, col, fn, decimals=3, cohort=None):
    """Apply `fn` to `col` within a dataset, with reference stats from the non-rasr methods plus each rasr cohort."""
    is_special = group['*algorithm*'].str.contains('rasr', case=False)
    out = pd.Series(index=group.index, dtype=float)
    base = group.loc[~is_special, col]
    out.loc[~is_special] = fn(base)

    special_idx = group.index[is_special]
    labels = pd.Series(special_idx, index=special_idx, dtype=object)
    if cohort is not None:
        given = cohort.reindex(special_idx)
        labels.loc[given.notna()] = given.loc[given.notna()]

    for _, idxs in labels.groupby(labels, sort=False).groups.items():
        idxs = list(idxs)
        combined = pd.concat([base, group.loc[idxs, col]])
        scored = fn(combined)
        for idx in idxs:
            out.loc[idx] = round(scored.loc[idx], decimals)
    return out

def r2_norm_excluding_similar(group, r2_col='r2_test', strategy=None, clip_negative=None, cohort=None):
    strategy = strategy or _r2_norm_strategy
    return _apply_excluding_similar(
        group, r2_col, lambda x: _normalize_r2(x, strategy, clip_negative), cohort=cohort)

def improvability_excluding_similar(group, err_col='rmse_test', cohort=None):
    return _apply_excluding_similar(group, err_col, _improvability, cohort=cohort)


In [ ]:
def compare(df_compare, x='r2_test',y='algorithm', row=None, col=None, scale=None, xlim=[],**kwargs):
    plt.figure()
    order = df_compare.groupby(y)[x].median().sort_values(ascending=False).index
    if scale=='log' and len(xlim)>0 and xlim[0] == 0:
        df_compare.loc[:,x] += 1
        xlim[0] = 1
        xnew = '1 + '+x
        df_compare=df_compare.rename(columns={x:xnew})
        x = xnew
    
    sns.catplot(data=df_compare, 
                kind='point',
                y=y,
                x=x,
                order=order,
                fliersize=0,
                row=row,
                col=col,
                **kwargs
               )

    plt.ylabel('')
    if len(xlim)>0:
        plt.xlim(xlim[0],xlim[1])
    if scale:
        plt.gca().set_xscale(scale)
    
    save(name='_'.join(['cat-boxplot',x + '-by-'+ y]) )
    if col:
        save(name='_'.join(['cat-boxplot',x + '-by-'+ y] + [col]) )



# excluded_datasets = ['1089_USCrime', '1193_BNG_lowbwt', '1199_BNG_echoMonths', '650_fri_c0_500_50']
# excluded_datasets = ['1089_USCrime', '1193_BNG_lowbwt', '1199_BNG_echoMonths', '650_fri_c0_500_50', '678_visualizing_environmental', '210_cloud', '192_vineyard']
excluded_datasets = []

excluded_methods = ['Linear']  # 'Linear' is only an imputation source; dropped from all plots and normalizations

alg_name_map = df_sum[['algorithm', '*algorithm*']].drop_duplicates().set_index('algorithm')['*algorithm*']

df_sum_filtered = df_sum[~df_sum['dataset'].isin(excluded_datasets)].copy()

df_sum_filtered['*algorithm*'] = df_sum_filtered['*algorithm*'].str.lstrip('*')


df_sum_filtered = df_sum_filtered[~df_sum_filtered['*algorithm*'].isin(excluded_methods)]

print(f"Remaining: {df_sum_filtered['dataset'].nunique()} datasets")
print(df_sum_filtered['dataset'].unique())

# Pareto plot

Plot the method results as trade-offs between accuracy and simplicity

In [ ]:
from pareto_utils import front
from matplotlib.transforms import Bbox
from matplotlib.lines import Line2D

labelsize=11
ticksize=11
smalllabelsize=11

# --- configure here ---
xcol          = 'model_size'  # x-axis column: a raw metric or its '*_rank'/'*_norm'/'*_improvability' variant
# '*_improvability': per-dataset relative headroom to the best method shown, (err - best_err) / err * 100,
# averaged across datasets (lower is better).
# NB: with _xscale='log' a method that is best on every dataset sits at 0 and is dropped; use 'linear' then.
ycol          = 'rmse_test_improvability'  # y-axis column: same options as xcol
_model_filter = r'^rasr_94m(_lambda[\d.]+)?$'  # keep only rasr methods matching this pattern (baselines unaffected); None keeps all
_seed_agg     = 'median'  # 'mean' or 'median' over the per-seed trials; datasets are always averaged with mean

_rename_methods = {
    'rasr_94m':            'RASR-94M-L',
    'rasr_94m_lambda0.15': 'RASR-94M-M',
    'rasr_94m_lambda0.5':  'RASR-94M-S',
    'PYSR':                'PySR',
}

_xscale  = 'log'     # 'linear' or 'log' scale for the x-axis
_yscale  = 'linear'  # 'linear' or 'log' scale for the y-axis
_x_invert = 'auto'  # True/False/'auto': flip the x-axis so 'better' points left
_y_invert = 'auto'  # same, for the y-axis (better points down)

_xlim = (4, 160)
_ylim = (2, 45)
_main_marker_s = 150
_marker_alpha = 0.8
_ps_tree_label_exception = True  # nudge PS-Tree's off-scale label so it clears the EQL marker
# ----------------------

# start from the per-seed df_results (df_sum is already collapsed over seeds)
data = df_results.copy()
data = data[~data['dataset'].isin(excluded_datasets)]
data['*algorithm*'] = data['algorithm'].map(alg_name_map).str.lstrip('*')
data = data[~data['*algorithm*'].isin(excluded_methods)]
data['rmse_test'] = np.sqrt(data['mse_test'])

# exclude candidate-count sweep variants ('..._c16', '..._c256')
data = data[~data['algorithm'].str.contains(r'_c\d+', regex=True, case=False)]

if _model_filter is not None:
    is_special = data['*algorithm*'].str.contains('rasr', case=False)
    data = data[~is_special | data['*algorithm*'].str.contains(_model_filter, case=False)]

def _base_metric(col):
    for suffix in ('_rank', '_norm', '_improvability'):
        if col.endswith(suffix):
            return col[:-len(suffix)]
    return col

metric_cols = sorted({_base_metric(col) for col in (xcol, ycol)} | {'r2_test'})
agg_fn = 'mean' if _seed_agg == 'mean' else 'median'
data = data.groupby(['*algorithm*', 'dataset'], as_index=False)[metric_cols].agg(agg_fn)

# cohort: the shown rasr methods form one reference set with the plain baselines, so variants shown
# together are scored against each other too. Built after the groupby-agg above (it resets the index).
if _model_filter is not None:
    _is_shown_special = data['*algorithm*'].str.contains('rasr', case=False)
    _cohort = pd.Series(np.where(_is_shown_special, 'shown', None), index=data.index)
else:
    _cohort = None

_metric_ascending = {
    'r2_test':           False,
    'r2_zero_test':      False,
    'model_size':        True,
    'mse_test':          True,
    'mae_test':          True,
    'rmse_test':         True,
    'log_mse_test':      True,
    'training time (s)': True,
}
# r2-like metrics are bounded by 1: on a log scale plot 1 - metric, so the best models land on the left
_metric_upper_bound = {'r2_test': 1.0, 'r2_zero_test': 1.0}

_short_metric_symbol = {
    'r2_test': '$R^2$',
    'r2_zero_test': '$R^2_0$',
    'model_size': 'Size',
    'training time (s)': 'Time',
    'rmse_test_improvability': 'Improvability',
}

def _rank_excluding_similar(group, metric_col, ascending):
    is_special = group['*algorithm*'].str.contains('rasr', case=False)
    ranks = pd.Series(index=group.index, dtype=float)
    base = group.loc[~is_special, metric_col]
    ranks.loc[~is_special] = base.rank(ascending=ascending)
    for idx in group.index[is_special]:
        combined = pd.concat([base, group.loc[[idx], metric_col]])
        ranks.loc[idx] = round(combined.rank(ascending=ascending).loc[idx], 3)
    return ranks

axis_scale = {xcol: _xscale, ycol: _yscale}
axis_ascending = {}
axis_metric = {}
for col in (xcol, ycol):
    is_rank = col.endswith('_rank')
    is_norm = col.endswith('_norm')
    is_improvability = col.endswith('_improvability')
    metric_col = _base_metric(col)
    ascending = _metric_ascending[metric_col]
    if is_rank:
        data[col] = data.groupby('dataset', group_keys=False).apply(
            lambda g: _rank_excluding_similar(g, metric_col, ascending=ascending))
        ascending = True
    elif is_norm:
        data[col] = data.groupby('dataset', group_keys=False).apply(
            lambda g: r2_norm_excluding_similar(
                g, r2_col=metric_col, cohort=(_cohort.reindex(g.index) if _cohort is not None else None)))
    elif is_improvability:
        data[col] = data.groupby('dataset', group_keys=False).apply(
            lambda g: improvability_excluding_similar(
                g, err_col=metric_col, cohort=(_cohort.reindex(g.index) if _cohort is not None else None)))
        ascending = True  # 0% = best method, 100% = infinitely improvable
    axis_ascending[col] = ascending
    axis_metric[col] = metric_col

# rename after the 'rasr'-substring logic above, which keys off the original names
pareto_data = data.groupby('*algorithm*')[[xcol, ycol]].mean()
pareto_data = pareto_data.rename(index=_rename_methods)

_r2_for_order = data.groupby('*algorithm*')['r2_test'].mean().rename(index=_rename_methods)

plot_data = pareto_data.copy()
axis_label_prefix = {xcol: '', ycol: ''}
_short_bound_prefix = {xcol: '', ycol: ''}
effective_ascending = dict(axis_ascending)
for col in (xcol, ycol):
    if axis_scale[col] == 'log' and axis_metric[col] in _metric_upper_bound:
        bound = _metric_upper_bound[axis_metric[col]]
        plot_data[col] = bound - plot_data[col]
        axis_label_prefix[col] = f'{bound:g} - '
        _short_bound_prefix[col] = f'{bound:g}-'
        effective_ascending[col] = True

def _to_plot_coords(col, vals):
    vals = np.asarray(vals, dtype=float)
    if axis_scale[col] == 'log' and axis_metric[col] in _metric_upper_bound:
        return _metric_upper_bound[axis_metric[col]] - vals
    return vals

# front() assumes lower is better on both objectives
objs = plot_data[[xcol,ycol]].values.copy()
for i, col in enumerate((xcol, ycol)):
    if not effective_ascending[col]:
        objs[:,i] = -objs[:,i]

PF = front(objs[:,0],objs[:,1])
pf_algorithms = set(plot_data.index[PF])

# one colour per method, from three 20-colour qualitative sets
_big_palette = (sns.color_palette('tab20', 20)
                + sns.color_palette('tab20b', 20)
                + sns.color_palette('tab20c', 20))
alg_colors = {alg: _big_palette[i % len(_big_palette)] for i, alg in enumerate(plot_data.index)}

# RASR variants get a green family; other methods too close to it move to the next free palette slot
_rasr_greens = {
    'RASR-94M-L': (0.192, 0.639, 0.329),
    'RASR-94M-M': (0.455, 0.769, 0.463),
    'RASR-94M-S': (0.631, 0.851, 0.608),
}
for _alg, _c in _rasr_greens.items():
    if _alg in alg_colors:
        alg_colors[_alg] = _c

def _color_dist(c1, c2):
    return sum((a - b) ** 2 for a, b in zip(c1, c2)) ** 0.5

_green_vals = list(_rasr_greens.values())
_used_colors = list(alg_colors.values())
_next_palette_i = len(alg_colors)
for _alg in list(alg_colors.keys()):
    if _alg in _rasr_greens:
        continue
    if any(_color_dist(alg_colors[_alg], _g) < 0.1 for _g in _green_vals):
        while (_big_palette[_next_palette_i % len(_big_palette)] in _used_colors
               or any(_color_dist(_big_palette[_next_palette_i % len(_big_palette)], _g) < 0.1
                      for _g in _green_vals)):
            _next_palette_i += 1
        _new_color = _big_palette[_next_palette_i % len(_big_palette)]
        alg_colors[_alg] = _new_color
        _used_colors.append(_new_color)
        _next_palette_i += 1

def _draw_pareto_plot(alg_colors, legend_algs, label_algs=None, others_color=None, others_label=None,
                       legend_loc='upper left', legend_bbox_to_anchor=(1.02, 1.0), legend_borderaxespad=0.,
                       figsize=(9, 7), direction_by_pareto=False, label_shift=None, bordered_algs=None,
                       legend_ncol=1, legend_split_after=None, font_pt=None, marker_scale=1.0,
                       box_aspect=None,
                       label_dir_override=None,  # {alg: (ddx, ddy)} unit direction for specific labels
                       xlim=None,
                       save_name='pareto_plot_improvability'):
    """Draw the Pareto scatter, front, legend and labels for a per-method colour map."""
    _label_algs = pf_algorithms if label_algs is None else set(label_algs)
    _label_shift = label_shift or {}  # {alg: (dx_px, dy_px)} manual nudge, on top of the automatic placement
    _label_dir_override = label_dir_override or {}
    _bordered_algs = set(plot_data.index) if bordered_algs is None else set(bordered_algs)
    _fs_axis = font_pt if font_pt is not None else labelsize
    _fs_small = font_pt if font_pt is not None else smalllabelsize
    _fs_legend = font_pt if font_pt is not None else 11
    _rc_overrides = {} if font_pt is None else {
        'font.size': font_pt, 'axes.titlesize': font_pt, 'axes.labelsize': font_pt,
        'xtick.labelsize': font_pt, 'ytick.labelsize': font_pt,
        'legend.fontsize': font_pt, 'legend.title_fontsize': font_pt,
    }
    _ms = marker_scale
    _eff_marker_s = _main_marker_s * _ms ** 2
    with plt.rc_context(_rc_overrides):
        fig = plt.figure(figsize=figsize)

        def _text_color(rgb):
            """Darken a light marker colour for use as label text."""
            luminance = 0.299 * rgb[0] + 0.587 * rgb[1] + 0.114 * rgb[2]
            return tuple(c * 0.55 for c in rgb) if luminance > 0.55 else rgb

        _scatter_colors = {alg: (*rgb, _marker_alpha) for alg, rgb in alg_colors.items()}

        # identified methods get a black border and a shape (square = neural, circle = other); the rest are plain circles
        _is_nsr_col = pd.Series([is_nsr(a) for a in plot_data.index], index=plot_data.index)
        _is_bordered_col = pd.Series([a in _bordered_algs for a in plot_data.index], index=plot_data.index)
        _square_s = _eff_marker_s * (np.pi / 4)
        ax = plt.gca()
        if box_aspect is not None:
            ax.set_box_aspect(box_aspect)
        _marker_groups = [
            (~_is_bordered_col & ~_is_nsr_col, 'o', 'white', 1.0 * _ms, _eff_marker_s),
            (~_is_bordered_col & _is_nsr_col, 's', 'white', 1.0 * _ms, _square_s),
            (_is_bordered_col & ~_is_nsr_col, 'o', '0.15', 1.0 * _ms, _eff_marker_s),
            (_is_bordered_col & _is_nsr_col, 's', '0.15', 1.0 * _ms, _square_s),
        ]
        for _mask, _marker, _edge, _lw, _s in _marker_groups:
            _sub = plot_data[_mask]
            if _sub.empty:
                continue
            sns.scatterplot(
                ax=ax, data=_sub, x=xcol, y=ycol, hue=_sub.index,
                palette={k: v for k, v in _scatter_colors.items() if k in _sub.index},
                marker=_marker, s=_s, legend=False, edgecolor=_edge, linewidth=_lw,
            )
        ax.set_zorder(2)

        if _xscale == 'log':
            ax.set_xscale('log')
        if _yscale == 'log':
            ax.set_yscale('log')

        x_invert = (not effective_ascending[xcol]) if _x_invert == 'auto' else _x_invert
        y_invert = (not effective_ascending[ycol]) if _y_invert == 'auto' else _y_invert
        if x_invert:
            ax.invert_xaxis()
        if y_invert:
            ax.invert_yaxis()

        _xlim_eff = xlim if xlim is not None else _xlim
        for _axis, _col, _rawlim, _inv in (('x', xcol, _xlim_eff, x_invert), ('y', ycol, _ylim, y_invert)):
            if _rawlim is None:
                continue
            _lo, _hi = sorted(_to_plot_coords(_col, _rawlim))
            if _inv:
                _lo, _hi = _hi, _lo
            getattr(ax, f'set_{_axis}lim')(_lo, _hi)

        if xcol.endswith('_rank') and ycol.endswith('_rank') and not x_invert and not y_invert and _xscale == _yscale == 'linear':
            ax.set_aspect(1.0)

        # dashed staircase Pareto front, built in `objs` space and mapped back to plot coordinates
        _sign_x = 1 if effective_ascending[xcol] else -1
        _sign_y = 1 if effective_ascending[ycol] else -1
        _front_obj = objs[PF][np.argsort(objs[PF, 0])]
        _step_x, _step_y = [], []
        for k, (ox, oy) in enumerate(_front_obj):
            if k > 0:
                _step_x.append(ox)
                _step_y.append(_front_obj[k - 1][1])
            _step_x.append(ox)
            _step_y.append(oy)
        _step_x = np.array(_step_x) * _sign_x
        _step_y = np.array(_step_y) * _sign_y

        # extend the staircase to the view edges
        _xlim_screen = ax.get_xlim()
        _ylim_screen = ax.get_ylim()
        _first_px_x = ax.transData.transform((_step_x[0], _step_y[0]))[0]
        _last_px_x = ax.transData.transform((_step_x[-1], _step_y[-1]))[0]
        if _first_px_x <= _last_px_x:
            _step_x = np.concatenate([[_step_x[0]], _step_x, [_xlim_screen[1]]])
            _step_y = np.concatenate([[_ylim_screen[1]], _step_y, [_step_y[-1]]])
        else:
            _step_x = np.concatenate([[_xlim_screen[1]], _step_x, [_step_x[-1]]])
            _step_y = np.concatenate([[_step_y[0]], _step_y, [_ylim_screen[1]]])

        ax.plot(_step_x, _step_y, '--', color='0.35', alpha=0.7, lw=1.3 * _ms, zorder=1)

        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_color('0.3')
            spine.set_linewidth(1.0 * _ms)

        ax.set_axisbelow(True)
        ax.xaxis.grid(True, which='major', color='0.85', linewidth=0.8 * _ms)
        # a log axis under one decade labels its ticks as minor ticks, so enable minor gridlines
        ax.xaxis.grid(True, which='minor', color='0.9', linewidth=0.6 * _ms)
        ax.yaxis.grid(True, which='major', color='0.85', linewidth=0.8 * _ms)
        ax.yaxis.grid(True, which='minor', color='0.9', linewidth=0.6 * _ms)

        # legend: Pareto front, shape key, optional 'Others', then one row per method (best test r2 first)
        _legend_order = [a for a in _r2_for_order.reindex(plot_data.index).sort_values(ascending=False).index
                          if a in legend_algs]
        _legend_handles = [Line2D([0], [0], linestyle='--', color='0.35', alpha=0.7, lw=1.3 * _ms, label='Pareto front')]
        _legend_handles.append(
            Line2D([0], [0], marker='s', linestyle='none', markersize=8.4 * _ms,
                   markerfacecolor='white', markeredgecolor='0.15', markeredgewidth=1.0 * _ms,
                   label='Amortized method'))
        _legend_handles.append(
            Line2D([0], [0], marker='o', linestyle='none', markersize=9.5 * _ms,
                   markerfacecolor='white', markeredgecolor='0.15', markeredgewidth=1.0 * _ms,
                   label='Non-amortized\nmethod'))
        if others_label is not None:
            _legend_handles.append(
                Line2D([0], [0], marker='o', linestyle='none', markersize=9.5 * _ms,
                       markerfacecolor=others_color, markeredgecolor='white', markeredgewidth=0.8 * _ms,
                       label=others_label))
        _legend_handles += [
            Line2D([0], [0], marker=('s' if is_nsr(alg) else 'o'), linestyle='none',
                   markersize=(8.4 if is_nsr(alg) else 9.5) * _ms,
                   markerfacecolor=alg_colors[alg], markeredgecolor='0.15', markeredgewidth=1.0 * _ms,
                   label=(f'{alg} *' if alg in pf_algorithms else alg))
            for alg in _legend_order
        ]
        if legend_ncol > 1 and legend_split_after is not None:
            # pad the handle list with blank entries so the column break lands after legend_split_after
            _split_row = next(i for i, h in enumerate(_legend_handles) if h.get_label() == legend_split_after) + 1
            _target_n = _split_row * legend_ncol
            while len(_legend_handles) < _target_n:
                _legend_handles.append(Line2D([0], [0], linestyle='none', label=''))
        legend = ax.legend(handles=_legend_handles, loc=legend_loc, bbox_to_anchor=legend_bbox_to_anchor,
                            fontsize=_fs_legend, frameon=True, borderaxespad=legend_borderaxespad,
                            labelspacing=0.425 * _ms, handletextpad=0.575 * _ms, ncol=legend_ncol)
        for _text in legend.get_texts():
            if is_rasr(_text.get_text()):
                _text.set_fontweight('bold')

        # tight_layout here so axes positions are final before pixel measurements
        fig.tight_layout()
        fig.canvas.draw()
        _axes_bbox = ax.get_window_extent()
        _legend_bbox = legend.get_window_extent()

        # labels: only Pareto-optimal in-view methods are named; a box-separation pass keeps them from overlapping

        # scatter `s` is an area, plot() `markersize` a diameter; convert so off-scale triangles match
        _offscale_ms = 2 * np.sqrt(_eff_marker_s / np.pi)
        _marker_radius_px = 12 * _ms
        _label_offset_px = 6 * _ms
        _offscale_offset_px = 8 * _ms
        _inner_margin_px = 4 * _ms
        _right_edge_margin_px = 14 * _ms

        def _pixel_shift(ax_, x, y, dxpix, dypix):
            """Return the data-coord point obtained by nudging (x, y) by a pixel offset."""
            xd, yd = ax_.transData.transform((x, y))
            return ax_.transData.inverted().transform((xd + dxpix, yd + dypix))

        def _mtv(bi, bj):
            """Minimum translation vector to move box `bi` clear of fixed box `bj`."""
            overlap_x = min(bi.x1, bj.x1) - max(bi.x0, bj.x0)
            overlap_y = min(bi.y1, bj.y1) - max(bi.y0, bj.y0)
            if overlap_x <= 0 or overlap_y <= 0:
                return 0.0, 0.0
            icx, icy = (bi.x0 + bi.x1) / 2, (bi.y0 + bi.y1) / 2
            jcx, jcy = (bj.x0 + bj.x1) / 2, (bj.y0 + bj.y1) / 2
            if overlap_x < overlap_y:
                return (overlap_x if icx >= jcx else -overlap_x), 0.0
            else:
                return 0.0, (overlap_y if icy >= jcy else -overlap_y)

        def _apply_lock(dx, dy, lock):
            """Zero out the axis a label isn't allowed to move along (see `locks` below)."""
            if lock == 'y':
                return dx, 0.0
            if lock == 'x':
                return 0.0, dy
            return dx, dy

        def _declutter_texts(fig_, ax_, texts, locks, obstacle_boxes, n_iter=80, pad=3.0):
            """Push label boxes clear of the obstacle boxes and of each other."""
            renderer = fig_.canvas.get_renderer()
            for _ in range(n_iter):
                boxes = [t.get_window_extent(renderer).padded(pad) for t in texts]
                moved = False
                for i in range(len(texts)):
                    for ob in obstacle_boxes:
                        dx, dy = _mtv(boxes[i], ob)
                        dx, dy = _apply_lock(dx, dy, locks[i])
                        if dx or dy:
                            moved = True
                            x, y = texts[i].get_position()
                            texts[i].set_position(_pixel_shift(ax_, x, y, dx, dy))
                            boxes[i] = texts[i].get_window_extent(renderer).padded(pad)
                for i in range(len(texts)):
                    for j in range(i + 1, len(texts)):
                        dx, dy = _mtv(boxes[i], boxes[j])
                        if dx == 0 and dy == 0:
                            continue
                        dxi, dyi = _apply_lock(dx, dy, locks[i])
                        dxj, dyj = _apply_lock(-dx, -dy, locks[j])
                        if dxi or dyi or dxj or dyj:
                            moved = True
                            x, y = texts[i].get_position()
                            texts[i].set_position(_pixel_shift(ax_, x, y, dxi, dyi))
                            x, y = texts[j].get_position()
                            texts[j].set_position(_pixel_shift(ax_, x, y, dxj, dyj))
                            boxes[i] = texts[i].get_window_extent(renderer).padded(pad)
                            boxes[j] = texts[j].get_window_extent(renderer).padded(pad)
                if not moved:
                    break

        _xmin, _xmax = sorted(ax.get_xlim())
        _ymin, _ymax = sorted(ax.get_ylim())

        _marker_rows = list(plot_data.iterrows())
        _marker_px = np.array([
            ax.transData.transform((min(max(row[xcol], _xmin), _xmax), min(max(row[ycol], _ymin), _ymax)))
            for _, row in _marker_rows
        ])

        _label_dir_candidates = [
            (0, 1), (0, -1), (1, 0), (-1, 0),
            (0.7071, 0.7071), (0.7071, -0.7071), (-0.7071, 0.7071), (-0.7071, -0.7071),
        ]

        def _best_label_dir(i):
            """Pick the compass direction whose label position is farthest from every other marker."""
            center = _marker_px[i]
            others = np.delete(_marker_px, i, axis=0)
            best_dir, best_score = (0, 1), -np.inf
            for dx, dy in _label_dir_candidates:
                cand = center + np.array([dx, dy]) * _label_offset_px
                score = np.min(np.hypot(*(others - cand).T)) if len(others) else 0.0
                if score > best_score:
                    best_score, best_dir = score, (dx, dy)
            return best_dir

        # triangle marker rotated per off-scale direction (4 cardinal + 4 diagonal)
        _arrow_angle = {
            (None, 'T'): 0, (None, 'B'): 180, ('R', None): -90, ('L', None): 90,
            ('R', 'T'): -45, ('L', 'T'): 45, ('R', 'B'): -135, ('L', 'B'): 135,
        }

        _texts, _anchors, _locks = [], [], []
        for i, (idx, row) in enumerate(_marker_rows):
            px, py = row[xcol], row[ycol]
            out_x = px < _xmin or px > _xmax
            out_y = py < _ymin or py > _ymax
            color = alg_colors[idx]

            if not out_x and not out_y:
                if idx not in _label_algs:
                    continue
                anchor = (px, py)
                if direction_by_pareto:
                    # direction_by_pareto: Pareto-optimal labels below their marker, others above, vertical movement only
                    ddx, ddy = (0, -1) if idx in pf_algorithms else (0, 1)
                    lock = 'x'
                    _dir_offset_px = _label_offset_px * (2.8 if idx in pf_algorithms else 3.6)
                elif idx in _label_dir_override:
                    ddx, ddy = _label_dir_override[idx]
                    lock = 'x' if ddx == 0 else ('y' if ddy == 0 else None)
                    _dir_offset_px = _label_offset_px
                else:
                    ddx, ddy = _best_label_dir(i)
                    lock = None
                    _dir_offset_px = _label_offset_px
                _shift_x, _shift_y = _label_shift.get(idx, (0, 0))
                label_xy = _pixel_shift(ax, px, py, ddx * _dir_offset_px + _shift_x, ddy * _dir_offset_px + _shift_y)
                label = idx
                fontsize = _fs_small
                fontweight = 'bold' if is_rasr(idx) else 'normal'
                bg = dict(facecolor='white', edgecolor='none', alpha=0.7, pad=1.5)
            else:
                cx = min(max(px, _xmin), _xmax)
                cy = min(max(py, _ymin), _ymax)
                h = 'R' if (out_x and px > _xmax) else ('L' if (out_x and px < _xmin) else None)
                v = 'T' if (out_y and py > _ymax) else ('B' if (out_y and py < _ymin) else None)
                marker = (3, 0, _arrow_angle[(h, v)])
                ddx = -_offscale_offset_px if h == 'R' else (_offscale_offset_px if h == 'L' else 0)
                ddy = -_offscale_offset_px if v == 'T' else (_offscale_offset_px if v == 'B' else 0)
                if _ps_tree_label_exception and idx == 'PS-Tree':
                    ddy += 14
                _shift_x, _shift_y = _label_shift.get(idx, (0, 0))
                ddx, ddy = ddx + _shift_x, ddy + _shift_y
                ax.plot([cx], [cy], marker=marker, color=color, ms=_offscale_ms,
                         mec=('0.15' if idx in _bordered_algs else 'white'), mew=1.1 * _ms,
                         zorder=4, clip_on=False)
                _bits = []
                if out_x:
                    _bits.append(f"{_short_bound_prefix[xcol]}{_short_metric_symbol.get(xcol, axis_metric[xcol])}={plot_data.loc[idx, xcol]:.3g}")
                if out_y:
                    _bits.append(f"{_short_bound_prefix[ycol]}{_short_metric_symbol.get(ycol, axis_metric[ycol])}={plot_data.loc[idx, ycol]:.3g}")
                anchor = (cx, cy)
                label_xy = _pixel_shift(ax, cx, cy, ddx, ddy)
                if others_label is not None and idx not in _bordered_algs:
                    label = ''
                elif idx not in legend_algs:
                    label = f'{idx}: ' + ', '.join(_bits)
                else:
                    label = ', '.join(_bits)
                fontsize, fontweight = _fs_small, 'normal'
                # off-scale on one axis: the label moves along that axis only
                if h is not None and v is None:
                    lock = 'y'
                elif v is not None and h is None:
                    lock = 'x'
                else:
                    lock = None
                bg = None

            if not label:
                continue

            _tcolor = _text_color(color) if (is_rasr(idx) or out_x or out_y) else '0.15'
            t = ax.text(*label_xy, label, ha='center', va='center', fontsize=fontsize,
                        color=_tcolor, fontweight=fontweight, zorder=6, clip_on=False, bbox=bg)

            if _ps_tree_label_exception and idx == 'PS-Tree':
                ax.plot([anchor[0], label_xy[0]], [anchor[1], label_xy[1]], color=color,
                        lw=0.9 * _ms, alpha=0.6, zorder=5, clip_on=False)
            _texts.append(t)
            _anchors.append(anchor)
            _locks.append(lock)

        fig.canvas.draw()

        _marker_boxes = [Bbox.from_bounds(cx - _marker_radius_px, cy - _marker_radius_px,
                                           2 * _marker_radius_px, 2 * _marker_radius_px)
                          for cx, cy in _marker_px]
        # direction_by_pareto skips decluttering so labels stay directly above/below their marker
        if not direction_by_pareto:
            _declutter_texts(fig, ax, _texts, _locks, _marker_boxes + [_legend_bbox.padded(6)])

        # final pass: keep label boxes inside the axes and (unless direction_by_pareto) clear of the legend
        _renderer = fig.canvas.get_renderer()
        for t in _texts:
            tb = t.get_window_extent(_renderer)
            half_w, half_h = (tb.x1 - tb.x0) / 2, (tb.y1 - tb.y0) / 2
            cx, cy = (tb.x0 + tb.x1) / 2, (tb.y0 + tb.y1) / 2
            x_lo = _axes_bbox.x0 + half_w + _inner_margin_px
            if direction_by_pareto:
                x_hi = _axes_bbox.x1 - half_w - _inner_margin_px
            else:
                x_hi = min(_axes_bbox.x1, _legend_bbox.x0) - half_w - _right_edge_margin_px
            y_lo = _axes_bbox.y0 + half_h + _inner_margin_px
            y_hi = _axes_bbox.y1 - half_h - _inner_margin_px
            new_cx = min(max(cx, x_lo), max(x_hi, x_lo))
            new_cy = min(max(cy, y_lo), max(y_hi, y_lo))
            if new_cx != cx or new_cy != cy:
                x, y = t.get_position()
                t.set_position(_pixel_shift(ax, x, y, new_cx - cx, new_cy - cy))

        _xlabel_map = {'r2_test': 'Mean Test $R^2$', 'r2_test_norm': 'Mean Normalized $R^2$',
                       'rmse_test_improvability': 'Mean Improvability (%)'}
        plt.xlabel('Mean Expression Size', fontsize=_fs_axis)
        plt.ylabel(axis_label_prefix[ycol] + _xlabel_map.get(ycol, ycol), fontsize=_fs_axis)

        save(name=save_name)

# for the small-datasets appendix variant (set with excluded_datasets and save_name above): flips the
# side of a few labels
_small_datasets_variant = False

_full_label_shift = {
    'FFX': (-45, 0),
    'PS-Tree': (-45, 10),
    'uDSR': (0, -15),
    'RASR-94M-M': (0, 12),
    'RASR-94M-S': (0, -12) if _small_datasets_variant else (0, 12),
    'RASR-94M-L': (0, 12) if _small_datasets_variant else (0, -12),
}
_full_label_dir_override = {
    'RASR-94M-M': (0, 1),
    'RASR-94M-S': (0, -1) if _small_datasets_variant else (0, 1),
    'RASR-94M-L': (0, 1) if _small_datasets_variant else (0, -1),
}
if _small_datasets_variant:
    _full_label_dir_override['Genetic Engine'] = (0, -1)

_draw_pareto_plot(alg_colors=alg_colors, legend_algs=list(plot_data.index),
                   legend_ncol=2, legend_split_after='AFP_FE',
                   box_aspect=1.0,
                   figsize=(5.55, 4.5),  # with font_pt, tuned so the tight crop is ~5.5in wide (the paper's \textwidth)
                   font_pt=7,
                   marker_scale=0.52,
                   label_shift=_full_label_shift,
                   label_dir_override=_full_label_dir_override,
                   save_name='pareto_plot_improvability')

# focus version for the paper intro: only the Pareto front plus PySR, Operon, SNIP, Flash-ANSR and E2E
# are identified; the rest is grey 'Others'
_focus_named = {'PySR', 'Operon', 'SNIP', 'Flash-ANSR', 'E2E'}
_focus_algs = pf_algorithms | {a for a in plot_data.index if a in _focus_named}
_others_grey = (0.75, 0.75, 0.75)

# colour-blind-safe palette (Okabe & Ito, 2008); green is reserved for RASR
_cb_palette = [
    (0.902, 0.624, 0.0),
    (0.337, 0.706, 0.914),
    (0.941, 0.894, 0.259),
    (0.0, 0.447, 0.698),
    (0.835, 0.369, 0.0),
    (0.800, 0.475, 0.655),
    (0.0, 0.0, 0.0),
]
_cb_green = (0.0, 0.620, 0.451)

_rasr_in_focus = sorted([a for a in _focus_algs if is_rasr(a)],
                         key=lambda a: (a != 'RASR-94M-L', a))
_other_in_focus = sorted(a for a in _focus_algs if not is_rasr(a))
alg_colors_focus = {}
for i, a in enumerate(_rasr_in_focus):
    if a in _rasr_greens:
        alg_colors_focus[a] = _rasr_greens[a]
    else:
        tint = (i / (len(_rasr_in_focus) - 1) * 0.6) if len(_rasr_in_focus) > 1 else 0.0
        alg_colors_focus[a] = tuple(c + (1 - c) * tint for c in _cb_green)
for i, a in enumerate(_other_in_focus):
    alg_colors_focus[a] = _cb_palette[i % len(_cb_palette)]
for a in plot_data.index:
    alg_colors_focus.setdefault(a, _others_grey)

_draw_pareto_plot(alg_colors=alg_colors_focus, legend_algs=[], label_algs=_focus_algs,
                   bordered_algs=_focus_algs,
                   others_color=_others_grey, others_label='Others',
                   legend_loc='upper right', legend_bbox_to_anchor=None, legend_borderaxespad=0.6, legend_ncol=1,
                   figsize=(5.0, 3.503),  # included at 0.9\linewidth (4.95in) in the paper, so fonts and markers keep their nominal size
                   font_pt=9,
                   marker_scale=0.7,
                   direction_by_pareto=True,
                   label_shift={
                       'Rils-Rols': (-14, -4),
                       'RASR-94M-L': (0, -12),
                       'RASR-94M-M': (0, -4),
                       'RASR-94M-S': (0, -4),
                       'Genetic Engine': (0, -4),
                   },
                   xlim=(4, 165),
                   save_name='pareto_plot_focus')

